# Mini-Projet Big Data — Sujet A : Avis clients e-commerce
Pipeline **Apache Spark → MongoDB** sur le dataset *Amazon Fine Food Reviews*.

Étapes : ingestion → nettoyage → 3 indicateurs → stockage NoSQL (MongoDB).

## 1. Initialisation de la SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("AvisEcommerce") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark

## 2. Ingestion des données
Le fichier contient des guillemets mal échappés dans certains avis : `multiLine` et `escape` corrigent le parsing.

In [ ]:
CHEMIN_CSV = "Reviews.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv(CHEMIN_CSV)

df.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



In [ ]:
# Aperçu du volume brut et du partitionnement initial
print("Nombre de lignes brutes :", df.count())
print("Nombre de partitions :", df.rdd.getNumPartitions())

Nombre de lignes brutes : 568454
Nombre de partitions : 1


## 3. Nettoyage des données
Trois règles imposées par le sujet : valeurs manquantes, doublons, notes hors échelle (1-5).

`repartition(4)` corrige le partitionnement initial à 1 seule partition, pour mieux paralléliser le traitement sur les cœurs disponibles.

In [ ]:
df_full_clean = df.dropna(subset=["ProductId", "Score", "Text"]) \
    .dropDuplicates() \
    .withColumn("Score", F.expr("try_cast(Score as float)")) \
    .filter((F.col("Score") >= 1) & (F.col("Score") <= 5)) \
    .repartition(4) \
    .cache()

print("Lignes apres nettoyage complet :", df_full_clean.count())

Lignes après nettoyage complet : 568454


### Test de robustesse du nettoyage
Le dataset réel ne contient ni valeur manquante, ni doublon, ni note hors échelle. Pour prouver que le pipeline fonctionne réellement, on injecte 3 lignes volontairement corrompues et on vérifie qu'elles sont bien supprimées.

In [ ]:
from pyspark.sql import Row

# On crée manuellement 3 lignes de données volontairement "sales",
# pour vérifier que notre pipeline de nettoyage les détecte et les supprime bien.
# Chaque Row doit respecter EXACTEMENT le même schéma (mêmes colonnes, mêmes types)
# que le DataFrame "df" original, sinon Spark refuse de les fusionner ensemble.
lignes_test = [

    # Ligne 1 : ProductId manquant (None)
    # → doit être supprimée par le dropna(subset=["ProductId", ...])
    Row(Id=999991, ProductId=None, UserId="U1", ProfileName="test",
        HelpfulnessNumerator=0, HelpfulnessDenominator=0,
        Score=3, Time=1300000000, Summary="test", Text="avis test"),

    # Ligne 2 : Score = 7, hors de l'échelle autorisée (1 à 5)
    # → doit être supprimée par le filter((Score >= 1) & (Score <= 5))
    # Note : Score est un entier (3, 7, 4), pas un float (3.0, 7.0, 4.0),
    # car Spark a détecté "Score" comme IntegerType dans le CSV d'origine
    Row(Id=999992, ProductId="B00TEST", UserId="U2", ProfileName="test",
        HelpfulnessNumerator=0, HelpfulnessDenominator=0,
        Score=7, Time=1300000000, Summary="test", Text="avis test"),

    # Ligne 3 : Text manquant (None)
    # → doit être supprimée par le dropna(subset=[..., "Text"])
    Row(Id=999993, ProductId="B00TEST2", UserId="U3", ProfileName="test",
        HelpfulnessNumerator=0, HelpfulnessDenominator=0,
        Score=4, Time=1300000000, Summary="test", Text=None),
]

# createDataFrame(..., schema=df.schema) : on force ces 3 lignes Python
# à respecter le même schéma (mêmes noms de colonnes, mêmes types) que "df",
# pour pouvoir les combiner avec les vraies données juste après
df_test = spark.createDataFrame(lignes_test, schema=df.schema)

# union(...) : empile les 3 lignes de test EN PLUS des vraies données,
# sans modifier "df" original (on travaille sur une copie "df_avec_test")
df_avec_test = df.union(df_test)

# On vérifie qu'on a bien ajouté exactement 3 lignes de plus qu'avant
print("Lignes avant nettoyage (avec les 3 lignes test) :", df_avec_test.count())

# On applique EXACTEMENT le même pipeline de nettoyage que sur les vraies données,
# pour vérifier qu'il fonctionne bien sur des cas volontairement problématiques
df_test_clean = df_avec_test.dropna(subset=["ProductId", "Score", "Text"]) \
    .dropDuplicates() \
    .withColumn("Score", F.expr("try_cast(Score as float)")) \
    .filter((F.col("Score") >= 1) & (F.col("Score") <= 5))

print("Lignes après nettoyage :", df_test_clean.count())

# Vérification finale : la différence entre "avant" et "après" doit être
# exactement 3 (les 3 lignes corrompues qu'on a injectées), pas plus, pas moins.
# Si c'est bien le cas, ça prouve que le nettoyage fonctionne correctement,
# et pas seulement "par absence de test" comme sur les vraies données
print("Les 3 lignes corrompues ont bien été supprimées :",
      df_avec_test.count() - df_test_clean.count() == 3)

Py4JJavaError: An error occurred while calling o162.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 32.0 failed 1 times, most recent failure: Lost task 7.0 in stage 32.0 (TID 57) (192.168.56.1 executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:303)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:182)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:330)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:73)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:57)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RD

## 4. Indicateur 1 — Note moyenne par produit

In [ ]:
stats_produits = df_full_clean.groupBy("ProductId").agg(
    F.round(F.avg("Score"), 2).alias("note_moyenne"),
    F.count("*").alias("nombre_avis")
).orderBy(F.desc("nombre_avis"))
        {
            "cell_type": "code",
            "id": "#VSC-1b236dcc",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# Construire 3 lignes de test en s'alignant sur l'ordre des colonnes de df.schema",
                "# (évite les problèmes de mismatch lors de la création et de l'union)",
                "cols = df.schema.names",
                "test_dicts = [",
                "    {\"Id\": 999991, \"ProductId\": None, \"UserId\": \"U1\", \"ProfileName\": \"test\",",
                "     \"HelpfulnessNumerator\": 0, \"HelpfulnessDenominator\": 0, \"Score\": 3, \"Time\": 1300000000, \"Summary\": \"test\", \"Text\": \"avis test\"},",
                "    {\"Id\": 999992, \"ProductId\": \"B00TEST\", \"UserId\": \"U2\", \"ProfileName\": \"test\",",
                "     \"HelpfulnessNumerator\": 0, \"HelpfulnessDenominator\": 0, \"Score\": 7, \"Time\": 1300000000, \"Summary\": \"test\", \"Text\": \"avis test\"},",
                "    {\"Id\": 999993, \"ProductId\": \"B00TEST2\", \"UserId\": \"U3\", \"ProfileName\": \"test\",",
                "     \"HelpfulnessNumerator\": 0, \"HelpfulnessDenominator\": 0, \"Score\": 4, \"Time\": 1300000000, \"Summary\": \"test\", \"Text\": None},",
                "]",
                "# Construire des tuples dans l'ordre exact des colonnes puis créer le DataFrame avec le même schema",
                "rows = [tuple(d.get(c) for c in cols) for d in test_dicts]",
                "df_test = spark.createDataFrame(rows, schema=df.schema)",
                "",
                "# Empiler les lignes de test sur les vraies données et vérifier le comptage",
                "df_avec_test = df.union(df_test)",
                "print(\"Lignes avant nettoyage (avec les 3 lignes test) :\", df_avec_test.count())",
                "",
                "# Appliquer le même pipeline de nettoyage que sur les vraies données",
                "df_test_clean = df_avec_test.dropna(subset=[\"ProductId\", \"Score\", \"Text\"]) \\",
                "    .dropDuplicates() \\",
                "    .withColumn(\"Score\", F.expr(\"try_cast(Score as float)\")) \\",
                "    .filter((F.col(\"Score\") >= 1) & (F.col(\"Score\") <= 5))",
                "",
                "print(\"Lignes après nettoyage :\", df_test_clean.count())",
                "print(\"Les 3 lignes corrompues ont bien été supprimées :\", df_avec_test.count() - df_test_clean.count() == 3)"
            ]
        },
print(client.list_database_names())

In [ ]:
# Ensemble des ProductId présents dans le top 20, pour enrichir les documents
top_ids = set(row["ProductId"] for row in top_produits.select("ProductId").collect())
print("Nombre de top produits :", len(top_ids))

In [ ]:
# Conversion en documents JSON (acceptable : résultat déjà agrégé et petit)
pdf_stats = stats_produits.toPandas()

documents = []
for _, row in pdf_stats.iterrows():
    documents.append({
        "_id": row["ProductId"],
        "note_moyenne": float(row["note_moyenne"]),
        "nombre_avis": int(row["nombre_avis"]),
        "est_top_produit": row["ProductId"] in top_ids
    })

print("Nombre de documents à insérer :", len(documents))
print(documents[0])

In [ ]:
collection = db["stats_produits"]
collection.drop()
resultat = collection.insert_many(documents)

print(f"{len(resultat.inserted_ids)} documents insérés dans 'stats_produits'")

In [ ]:
print("Nombre de documents dans la collection :", collection.count_documents({}))
print(collection.find_one())

In [ ]:
pdf_evolution = evolution_avis.toPandas()

documents_evolution = []
for _, row in pdf_evolution.iterrows():
    documents_evolution.append({
        "annee_mois": row["annee_mois"],
        "nombre_avis": int(row["nombre_avis"])
    })

collection_evolution = db["evolution_avis"]
collection_evolution.drop()
resultat_evo = collection_evolution.insert_many(documents_evolution)

print(f"{len(resultat_evo.inserted_ids)} documents insérés dans 'evolution_avis'")